# ODE vs SDE Roughness

Chapter 19: Stochastic Differential Equations · *Mathematics for AI and Machine Learning* by Fuheng Wu

Zoom into a smooth sine curve vs a Brownian path.

Companion code for the [interactive demo](https://math4ai.org/companion/chapter-19/ode-vs-sde/) on math4ai.org (sign-in required). Run the cell with Shift+Enter; PyTorch is preinstalled on Colab. Change the values at the top and run it again to explore.


In [1]:
import math

import torch

torch.manual_seed(42)

# A smooth ODE path vs a Brownian path on [0, 2], and what happens when you zoom in (demo zoom 4)
T, zoom = 2.0, 4


def brownian(dt, T=T):
    n = round(T / dt)
    return torch.cat([torch.zeros(1, dtype=torch.float64), (math.sqrt(dt) * torch.randn(n, dtype=torch.float64)).cumsum(0)])


# ── Smooth path: differences shrink like dt, so slopes settle down ──────────
for dt in (1e-2, 1e-3, 1e-4):
    t = torch.arange(0, T + dt / 2, dt, dtype=torch.float64)
    x = torch.exp(-t)                                   # solves dx/dt = -x
    print(f"ODE  dt={dt:.0e}: max |slope| = {((x[1:] - x[:-1]) / dt).abs().max().item():8.3f}, "
          f"total variation = {(x[1:] - x[:-1]).abs().sum().item():.4f}")

# ── Brownian path: increments shrink like sqrt(dt), so "slopes" blow up ─────
for dt in (1e-2, 1e-3, 1e-4):
    w = brownian(dt)
    d = w[1:] - w[:-1]
    print(f"BM   dt={dt:.0e}: max |slope| = {(d / dt).abs().max().item():8.1f}, "
          f"total variation = {d.abs().sum().item():7.2f}, quadratic variation = {(d ** 2).sum().item():.4f} (T = {T})")

# ── Self-similarity: W(c t) has the law of sqrt(c) W(t) ─────────────────────
# Zooming into a window zoom times shorter shows fluctuations sqrt(zoom) times smaller.
paths = (math.sqrt(T / 1000) * torch.randn(20000, 1000, dtype=torch.float64)).cumsum(1)
full, window = paths[:, -1], paths[:, 1000 // zoom - 1]       # W(T) and W(T / zoom)
print(f"\nstd of W(T) / std of W(T/{zoom}) over 20000 paths = {(full.std() / window.std()).item():.3f}  "
      f"(sqrt({zoom}) = {math.sqrt(zoom):.3f})")

ODE  dt=1e-02: max |slope| =    0.995, total variation = 0.8647
ODE  dt=1e-03: max |slope| =    1.000, total variation = 0.8647
ODE  dt=1e-04: max |slope| =    1.000, total variation = 0.8647
BM   dt=1e-02: max |slope| =     27.2, total variation =   16.70, quadratic variation = 2.2132 (T = 2.0)
BM   dt=1e-03: max |slope| =    141.4, total variation =   50.23, quadratic variation = 1.9792 (T = 2.0)
BM   dt=1e-04: max |slope| =    454.9, total variation =  159.74, quadratic variation = 2.0073 (T = 2.0)



std of W(T) / std of W(T/4) over 20000 paths = 2.017  (sqrt(4) = 2.000)
